# U2-1 — Clinical BERT as a Domain Feature Extractor

**MSDS 565 · Unit 2 (Text Data) · Demo**

We *don't* fine-tune anything here and we *don't* do masked-language modeling.
Instead we ask a single, measurable question:

> Holding the classifier fixed, does **domain pretraining** give us a better
> document representation for a real clinical task?

To answer it we classify medical transcriptions by **specialty** using three
representations and one classifier:

| Representation | Source |
|---|---|
| TF-IDF | classic sparse vectorization (Unit 1 §1 callback) |
| general BERT embeddings | `bert-base-cased`, frozen |
| **clinical** BERT embeddings | `emilyalsentzer/Bio_ClinicalBERT`, frozen |

Everything runs through the high-level `transformers` **`feature-extraction`
pipeline** — no `torch`, no manual forward passes.

### CRISP-DM map
Data Understanding → Representation → Modeling → Evaluation → Interpretation.

### Learning objectives
- Treat a pretrained transformer as a *frozen feature extractor*.
- Compare representations on a fixed downstream task (controlled comparison).
- Read macro-F1 under class imbalance; tie back to Unit 1 imbalanced classification.


## Setup

The clinical model descends from BioBERT and is **cased**, so the fair
general-domain baseline is `bert-base-cased` (same vocab family), not the
uncased variant — otherwise we'd be measuring tokenization differences, not
pretraining.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

plt.style.use("dark_background")

CLINICAL_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
GENERAL_MODEL  = "bert-base-cased"

# Download the Kaggle "Medical Transcriptions" dataset (mtsamples-derived).
# Expected columns: 'transcription', 'medical_specialty'. No credentialing needed.
MTSAMPLES_CSV     = "mtsamples.csv"
TOP_K_SPECIALTIES = 6     # keep the most populous specialties
MAX_PER_CLASS     = 120   # cap per class -> classroom-runnable on CPU
MAX_LEN           = 256   # truncate long notes for embedding
TEST_SIZE         = 0.25
RANDOM_STATE      = 42

## 1. Data understanding

MTSamples is messy on purpose — dozens of specialties, heavy imbalance, some
catch-all labels. We keep the `TOP_K_SPECIALTIES` most populous classes and cap
each at `MAX_PER_CLASS` so the demo runs in class on CPU. (Scaling up is a good
homework extension.)

In [ ]:
def load_mtsamples(path, top_k, cap, seed=RANDOM_STATE):
    df = pd.read_csv(path)[["transcription", "medical_specialty"]].dropna()
    df["medical_specialty"] = df["medical_specialty"].str.strip()
    keep = df["medical_specialty"].value_counts().head(top_k).index
    df = df[df["medical_specialty"].isin(keep)]
    df = (df.groupby("medical_specialty", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), cap), random_state=seed)))
    return df.reset_index(drop=True)

df = load_mtsamples(MTSAMPLES_CSV, TOP_K_SPECIALTIES, MAX_PER_CLASS)
print(f"{len(df)} documents across {df['medical_specialty'].nunique()} specialties\n")
print(df["medical_specialty"].value_counts())

In [ ]:
texts = df["transcription"].tolist()
y     = df["medical_specialty"].to_numpy()

idx = np.arange(len(df))
idx_tr, idx_te, y_tr, y_te = train_test_split(
    idx, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print(f"train={len(idx_tr)}  test={len(idx_te)}")

## 2. Representation A — TF-IDF baseline

The bag-of-words baseline students already know. Fit the vectorizer on the
**training split only**, then transform the test split.

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                        sublinear_tf=True, stop_words="english")
Xtr_tfidf = tfidf.fit_transform([texts[i] for i in idx_tr])
Xte_tfidf = tfidf.transform([texts[i] for i in idx_te])
print("TF-IDF train matrix:", Xtr_tfidf.shape)

## 3. Representations B & C — frozen transformer embeddings

The `feature-extraction` pipeline returns one contextual vector per token. We
mean-pool those into a single document vector,

$$\mathbf{d} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{h}_t,$$

with no fine-tuning and no `torch`. Long notes are truncated to `MAX_LEN`
tokens. This is the slow cell — embedding is a forward pass per document on CPU.

In [ ]:
def embed_texts(model_name, texts, max_len=MAX_LEN, log_every=25):
    pipe = pipeline("feature-extraction", model=model_name)
    vecs = []
    for i, t in enumerate(texts):
        out = np.array(pipe(t, tokenize_kwargs={"truncation": True,
                                                "max_length": max_len}))[0]
        vecs.append(out.mean(axis=0))
        if (i + 1) % log_every == 0:
            print(f"  [{model_name.split('/')[-1]}] {i + 1}/{len(texts)}")
    return np.vstack(vecs)

In [ ]:
# Embed every document once, then index by the split (embeddings are frozen,
# so they don't depend on train/test).
E_general  = embed_texts(GENERAL_MODEL, texts)
E_clinical = embed_texts(CLINICAL_MODEL, texts)
print("embedding dim:", E_general.shape[1])

## 4. Modeling & evaluation

Same classifier everywhere: multinomial `LogisticRegression` with
`class_weight="balanced"` to respect the imbalance. Dense embeddings get a
`StandardScaler` (fit on train); sparse TF-IDF is left as-is. We report accuracy
and **macro-F1**,

$$\text{F1}_{\text{macro}} = \frac{1}{C}\sum_{c=1}^{C}\frac{2\,P_c R_c}{P_c + R_c},$$

which weights every specialty equally regardless of frequency.

In [ ]:
def fit_eval(Xtr, Xte, y_tr, y_te, name, scale=False):
    if scale:
        sc = StandardScaler()
        Xtr, Xte = sc.fit_transform(Xtr), sc.transform(Xte)
    clf = LogisticRegression(max_iter=3000, class_weight="balanced")
    clf.fit(Xtr, y_tr)
    pred = clf.predict(Xte)
    acc = accuracy_score(y_te, pred)
    f1  = f1_score(y_te, pred, average="macro")
    print(f"{name:14s}  acc={acc:.3f}  macro-F1={f1:.3f}")
    return {"name": name, "acc": acc, "f1": f1, "pred": pred}

results = [
    fit_eval(Xtr_tfidf, Xte_tfidf, y_tr, y_te, "TF-IDF"),
    fit_eval(E_general[idx_tr],  E_general[idx_te],  y_tr, y_te, "general BERT", scale=True),
    fit_eval(E_clinical[idx_tr], E_clinical[idx_te], y_tr, y_te, "Clinical BERT", scale=True),
]

### Representation comparison

In [ ]:
names = [r["name"] for r in results]
accs  = [r["acc"] for r in results]
f1s   = [r["f1"] for r in results]

x = np.arange(len(names))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, accs, w, label="accuracy", color="#4ec9b0")
ax.bar(x + w/2, f1s,  w, label="macro-F1", color="#c586c0")
ax.set_xticks(x, names)
ax.set_ylim(0, 1)
ax.set_ylabel("score")
ax.set_title("Same classifier, three representations")
ax.legend()
for xi, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(xi - w/2, a + 0.01, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(xi + w/2, f + 0.01, f"{f:.2f}", ha="center", fontsize=8)
fig.tight_layout()
plt.show()

### Where Clinical BERT helps (and where it doesn't)

The confusion matrix shows *which* specialties the clinical representation
separates — often the ones whose vocabulary is distinctive (cardiology,
radiology) versus catch-all labels that overlap everything.

In [ ]:
best = max(results, key=lambda r: r["f1"])
labels = sorted(np.unique(y_te))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_te, best["pred"], labels=labels, xticks_rotation=45,
    cmap="magma", colorbar=False, ax=ax)
ax.set_title(f"Confusion matrix — {best['name']}")
fig.tight_layout()
plt.show()

print(classification_report(y_te, best["pred"]))

## 5. Discussion & extensions

**The teaching point.** The classifier never changed. TF-IDF knows nothing
beyond word counts; general BERT knows English; Clinical BERT was further
pretrained on clinical notes. Any macro-F1 gap is the *value of domain
pretraining*, isolated cleanly.

**Honest caveats to raise in class:**
- Frozen + mean-pooled embeddings *undersell* BERT. Fine-tuning typically beats
  this, but that's a separate (slower) lesson. The point here is representation,
  not state-of-the-art.
- TF-IDF is a surprisingly strong baseline when specialty vocabulary is
  distinctive — a healthy reminder not to reach for transformers reflexively.

**Extensions / homework hooks:**
1. *Imbalance (Unit 1):* swap `class_weight` for SMOTE on the embeddings; compare.
2. *Scale up:* raise `MAX_PER_CLASS` / `TOP_K_SPECIALTIES`; watch macro-F1 move.
3. *Pooling:* try `[CLS]` pooling vs. mean pooling.
4. *Topic modeling:* feed `E_clinical` into **BERTopic** (`embedding_model`) for an
   unsupervised view of the same corpus — fits Unit 2's BERTopic section.
5. *Fairness/interpretability (Unit 1):* run SHAP on the TF-IDF classifier to see
   which tokens drive each specialty.
